# Manual turn-ban path check (AequilibraE vs NetworkX)

This notebook does one focused exercise:
1. Sample 1000 OD pairs
2. Compute baseline AequilibraE paths
3. Pick one random consecutive 2-link sequence (with directions) per path
4. Add all selected sequences as turn bans
5. Recompute paths for the same OD pairs with AequilibraE and NetworkX
6. Compare only the sequence of link IDs

In [2]:
from pathlib import Path
from collections import defaultdict

import networkx as nx
import numpy as np
import pandas as pd

from aequilibrae import Project
from tqdm.auto import tqdm

# ---- USER INPUTS ----
# MODEL_PATH = Path(r"D:\release\Sample models\new_chicago\chicago_sample_model")
MODEL_PATH = Path(r"D:\release\coquimbo")
# MODEL_PATH = Path(r"D:\release\Arkansas\model")
MODE = "c"
COST_FIELD = "distance"
SAMPLE_SIZE = 1
SEED = 42
ALLOW_UTURNS = False

rng = np.random.default_rng(SEED)

In [2]:
def sample_od_pairs(centroids, sample_size, rng):
    if len(centroids) < 2:
        raise ValueError("Need at least 2 centroids")

    origins = rng.choice(centroids, size=sample_size, replace=True)
    destinations = rng.choice(centroids, size=sample_size, replace=True)

    same = origins == destinations
    while np.any(same):
        destinations[same] = rng.choice(centroids, size=int(np.sum(same)), replace=True)
        same = origins == destinations

    return [(int(o), int(d)) for o, d in zip(origins, destinations)]


def build_arc_metadata(graph, cost_field):
    gdf = graph.graph[["id", "a_node", "b_node", "link_id", "direction", cost_field]].copy()
    gdf["a_node"] = graph.all_nodes[gdf["a_node"].to_numpy(dtype=np.int64)]
    gdf["b_node"] = graph.all_nodes[gdf["b_node"].to_numpy(dtype=np.int64)]
    gdf = gdf.rename(columns={cost_field: "cost"})

    arc_meta = {}
    incoming = {}
    outgoing = {}

    for row in gdf.itertuples(index=False):
        arc_id = int(row.id)
        a_node = int(row.a_node)
        b_node = int(row.b_node)
        arc_meta[arc_id] = {
            "a_node": a_node,
            "b_node": b_node,
            "link_id": int(row.link_id),
            "direction": int(row.direction),
            "cost": float(row.cost),
        }
        outgoing.setdefault(a_node, []).append(arc_id)
        incoming.setdefault(b_node, []).append(arc_id)

    return gdf, arc_meta, incoming, outgoing


def build_state_graph(arc_meta, incoming, outgoing, prohibited_turns, allow_uturns=False):
    prohibited = set(
        tuple(int(v) for v in row)
        for row in prohibited_turns[["from_node", "via_node", "to_node"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    g = nx.DiGraph()
    for node in set(incoming.keys()) & set(outgoing.keys()):
        for in_arc in incoming[node]:
            in_lnk = arc_meta[in_arc]
            for out_arc in outgoing[node]:
                out_lnk = arc_meta[out_arc]

                if not allow_uturns:
                    if int(out_lnk["b_node"]) == int(in_lnk["a_node"]):
                        continue

                turn = (
                    int(in_lnk["a_node"]),
                    int(in_lnk["b_node"]),
                    int(out_lnk["b_node"]),
                )
                if turn in prohibited:
                    continue

                g.add_edge(in_arc, out_arc, weight=out_lnk["cost"])

    return g


def nx_path_for_od(state_graph, arc_meta, outgoing, incoming, orig, dest):
    start_arcs = outgoing.get(orig, [])
    end_arcs = incoming.get(dest, [])

    if not start_arcs or not end_arcs:
        return None

    source = ("source", orig, dest)
    sink = ("sink", orig, dest)

    g = state_graph.copy()
    for arc in start_arcs:
        g.add_edge(source, arc, weight=arc_meta[arc]["cost"])
    for arc in end_arcs:
        g.add_edge(arc, sink, weight=0.0)

    try:
        state_path = nx.shortest_path(g, source=source, target=sink, weight="weight")
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None

    arc_sequence = [x for x in state_path if isinstance(x, (int, np.integer))]
    link_sequence = [int(arc_meta[a]["link_id"]) for a in arc_sequence]
    direction_sequence = [int(arc_meta[a]["direction"]) for a in arc_sequence]

    if len(arc_sequence) == 0:
        node_sequence = []
    else:
        node_sequence = [int(arc_meta[arc_sequence[0]]["a_node"])] + [int(arc_meta[a]["b_node"]) for a in arc_sequence]

    return link_sequence, direction_sequence, node_sequence

In [3]:
project = Project.from_path(MODEL_PATH)
project.upgrade()
project.network.build_graphs(modes=[MODE])
graph = project.network.graphs[MODE]
graph.set_graph(COST_FIELD)
graph.set_blocked_centroid_flows(False)

centroids = np.asarray(graph.centroids, dtype=np.int64)
if centroids.size == 0:
    raise RuntimeError("No centroids found in graph")

od_pairs = sample_od_pairs(centroids, SAMPLE_SIZE, rng)

Ignoring results database during upgrade


In [4]:
selected_turns = []
rows_for_ban_selection = []
for orig, dest in tqdm(od_pairs, desc="AEQ baseline paths", total=len(od_pairs)):
    res = graph.compute_path(orig, dest)
    if res.path_nodes is None or res.path_nodes.shape[0] < 5:  # Need at least one interior node triple
        continue

    nodes = [int(x) for x in res.path_nodes]

    idx = int(rng.integers(1, len(nodes) - 2))
    turn = (nodes[idx - 1], nodes[idx], nodes[idx + 1])
    selected_turns.append(turn)

    rows_for_ban_selection.append([orig, dest, turn[0], turn[1], turn[2]])

cols = ["orig", "dest", "from_node", "via_node", "to_node"]
selected_turns_df = pd.DataFrame(rows_for_ban_selection, columns=cols)

selected_turns_df.drop_duplicates(subset=["from_node", "via_node", "to_node"], inplace=True)
turn_bans = selected_turns_df.assign(penalty=np.nan)

if turn_bans.empty:
    raise RuntimeError("No valid node-based turns were collected for bans")

graph.set_turn_restrictions(turn_bans, allow_path_uturns=ALLOW_UTURNS)

print(f"Sampled OD pairs: {len(od_pairs)}")
print(f"Successful baseline paths: {selected_turns_df.shape[0]}")
print(f"Unique prohibited turns added: {len(turn_bans)}")

AEQ baseline paths: 100%|██████████| 2000/2000 [00:04<00:00, 496.69it/s]


Sampled OD pairs: 2000
Successful baseline paths: 1648
Unique prohibited turns added: 1648


In [5]:
graph_df, arc_meta, incoming, outgoing = build_arc_metadata(graph, COST_FIELD)
state_graph = build_state_graph(
    arc_meta=arc_meta,
    incoming=incoming,
    outgoing=outgoing,
    prohibited_turns=turn_bans,
    allow_path_uturns=ALLOW_UTURNS,
)

In [6]:
def check_path_turns(path_nodes, turn_bans):
    if path_nodes is None or len(path_nodes) < 3:
        return

    turns = pd.DataFrame(
        {
            "from_node": path_nodes[:-2],
            "via_node": path_nodes[1:-1],
            "to_node": path_nodes[2:],
        }
    )
    idx1 = pd.MultiIndex.from_frame(turns[["from_node", "via_node", "to_node"]])
    idx2 = pd.MultiIndex.from_frame(turn_bans.drop_duplicates()[["from_node", "via_node", "to_node"]])
    assert turns[idx1.isin(idx2)].empty

In [7]:
comparison_rows = []

links = project.network.links.data[["link_id", "distance"]].copy().set_index("link_id")

for orig, dest in tqdm(od_pairs, desc="Comparing AEQ vs NX", total=len(od_pairs)):
    aeq_links = None
    aeq_dirs = None
    aeq_nodes = None
    nx_links = None
    nx_dirs = None
    nx_nodes = None
    status = "ok"

    try:
        res = graph.compute_path(orig, dest)
        aeq_links = [int(x) for x in res.path]
        aeq_dirs = [int(x) for x in res.path_link_directions]
        aeq_nodes = [int(x) for x in res.path_nodes]
    except Exception:
        status = "no_path_found"

    nx_result = nx_path_for_od(state_graph, arc_meta, outgoing, incoming, orig, dest)
    if nx_result is None:
        if status == "ok":
            status = "nx_unreachable"
    else:
        nx_links, nx_dirs, nx_nodes = nx_result

    if aeq_links is None and nx_links is not None:
        status = "ONLY AEQ FAILED"
    elif aeq_links is not None and nx_links is None:
        status = "ONLY NETWORKX FAILED"

    link_sequence_match = bool(aeq_links == nx_links) if aeq_links is not None and nx_links is not None else True

    diff_type = ""
    if not link_sequence_match:
        if nx_links is not None and aeq_links is not None:
            if set(aeq_links) == set(nx_links):
                diff_type = "Different order. Around the block?"
            else:
                diff_type = "DIFFERENT PATH"

    nx_dist = float(links.query("link_id in @nx_links").distance.sum()) if nx_links is not None else -1
    aeq_dist = float(links.query("link_id in @aeq_links").distance.sum()) if aeq_links is not None else -1
    comparison_rows.append(
        {
            "orig": orig,
            "dest": dest,
            "status": status,
            "link_sequence_match": link_sequence_match,
            "Difference Type": diff_type,
            "aeq_num_links": None if aeq_links is None else len(aeq_links),
            "nx_num_links": None if nx_links is None else len(nx_links),
            "aeq_distance": None if aeq_links is None else aeq_dist,
            "nx_distance": None if nx_links is None else nx_dist,
            "aeq_links": aeq_links,
            "nx_links": nx_links,
            "aeq_link_dirs": aeq_dirs,
            "nx_link_dirs": nx_dirs,
            "aeq_nodes": aeq_nodes,
            "nx_nodes": nx_nodes,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
ok_df = comparison_df[comparison_df["status"] == "ok"]

print(f"Compared OD pairs: {len(comparison_df)}")
print(f"Rows with valid paths in both: {len(ok_df)}")
if len(ok_df):
    print(f"Exact link-sequence matches: {int(ok_df['link_sequence_match'].sum())}/{len(ok_df)}")

mismatch_df = ok_df[~ok_df["link_sequence_match"]].copy()
print(f"Mismatches to inspect manually: {len(mismatch_df)}")

Comparing AEQ vs NX: 100%|██████████| 2000/2000 [11:22<00:00,  2.93it/s]

Compared OD pairs: 2000
Rows with valid paths in both: 264
Exact link-sequence matches: 259/264
Mismatches to inspect manually: 5


In [8]:
mismatch_df

,orig,dest,status,link_sequence_match,Difference Type,aeq_num_links,nx_num_links,aeq_distance,nx_distance,aeq_links,nx_links,aeq_link_dirs,nx_link_dirs,aeq_nodes,nx_nodes
89,111,3,ok,False,Different order. Around the block?,377.0,377.0,37752.686660,37752.686660,"[34919, 11007, 23630, 23629, 23628, 5656, 2446...","[34919, 11007, 23630, 23629, 24461, 5656, 2362...","[1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, ...","[1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1,...","[111, 44380, 51476, 79843, 69496, 51465, 27310...","[111, 44380, 51476, 79843, 69496, 27310, 51465..."
333,111,110,ok,False,Different order. Around the block?,35.0,35.0,1677.643862,1677.643862,"[34919, 11007, 23630, 23629, 23628, 5656, 2446...","[34919, 11007, 23630, 23629, 24461, 5656, 2362...","[1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, ...","[1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1,...","[111, 44380, 51476, 79843, 69496, 51465, 27310...","[111, 44380, 51476, 79843, 69496, 27310, 51465..."
544,111,108,ok,False,Different order. Around the block?,86.0,86.0,3984.556582,3984.556582,"[34919, 11007, 23630, 23629, 23628, 5656, 2446...","[34919, 11007, 23630, 23629, 24461, 5656, 2362...","[1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, ...","[1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1,...","[111, 44380, 51476, 79843, 69496, 51465, 27310...","[111, 44380, 51476, 79843, 69496, 27310, 51465..."
1039,111,27,ok,False,Different order. Around the block?,290.0,290.0,20407.208645,20407.208645,"[34919, 11007, 23630, 23629, 23628, 5656, 2446...","[34919, 11007, 23630, 23629, 24461, 5656, 2362...","[1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, ...","[1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1,...","[111, 44380, 51476, 79843, 69496, 51465, 27310...","[111, 44380, 51476, 79843, 69496, 27310, 51465..."
1929,127,99,ok,False,Different order. Around the block?,31.0,31.0,2698.717329,2698.717329,"[34936, 24378, 24379, 24380, 24157, 24158, 241...","[34936, 24378, 24379, 24380, 24157, 24158, 241...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1...","[127, 74408, 47012, 23623, 69419, 79667, 39220...","[127, 74408, 47012, 23623, 69419, 79667, 39220..."


In [ ]:
check_path_turns(mismatch_df.nx_nodes.values[0], turn_bans)
check_path_turns(mismatch_df.aeq_nodes.values[0], turn_bans)

In [ ]:
mismatch_df.assign(difference=mismatch_df.aeq_distance - mismatch_df.nx_distance)

In [ ]:
graph.compute_path(1584, 968).milepost

In [ ]:
project.close()
print("Project closed")